# Social Media Processing

This notebook shows how to use HuggingFace's package to import and train regression models to assess humor rating in social media posts in English (SemEval2021: HaHackathon: Detecting and Rating Humor and Offense https://competitions.codalab.org/competitions/27446, **Task-1b**).

Detection of humour, especially in social media posts, poses a linguistic challenge to NLP, due to the noise, figurative language, contextuality and subjectivity. You will hence try different methods to address those challenges such as preprocessing, data augmentation, ensembling and multi-task learning.

We will download and unzip the data from here: http://smash.inf.ed.ac.uk/hahackathon_data/hahackathon_data.zip.


We recommend you to do this lab on a Colab GPU provided by Google.

In [ ]:
!wget http://smash.inf.ed.ac.uk/hahackathon_data/hahackathon_data.zip
!unzip '/content/hahackathon_data.zip' -x __MACOSX/*

--2026-03-14 21:36:18--  http://smash.inf.ed.ac.uk/hahackathon_data/hahackathon_data.zip
Resolving smash.inf.ed.ac.uk (smash.inf.ed.ac.uk)... 129.215.10.97
Connecting to smash.inf.ed.ac.uk (smash.inf.ed.ac.uk)|129.215.10.97|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 571603 (558K) [application/zip]
Saving to: ‘hahackathon_data.zip.3’

hahackathon_data.zi 100%[===================>] 558.21K   788KB/s    in 0.7s    

2026-03-14 21:36:19 (788 KB/s) - ‘hahackathon_data.zip.3’ saved [571603/571603]

Archive:  /content/hahackathon_data.zip
replace hahackathon data/.DS_Store? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

First, we need to install Hugging Face [transformers](https://huggingface.co/transformers/index.html) and [Sentence piece Tokenizers](https://github.com/google/sentencepiece), as well as some helper libraries, with the following commands.

In [ ]:
!pip install -q transformers
!pip install -q sentencepiece
!pip install -q  ipywidgets
!jupyter nbextension enable --py widgetsnbextension

In [ ]:
import numpy as np
import random
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from transformers import DistilBertTokenizer, DistilBertModel
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader

We define the fix seed method to be able to introduce variety into ensembling models.

In [ ]:
# Set random seed for reproducibility
def set_random_seed(seed=123):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_random_seed()

## Regression with BERT

We will use the [DistilBert](https://arxiv.org/abs/1910.01108v4) model and its Tokeniser following the preprocessing code from Lab on Aspect-Based Sentiment Analysis.

In [ ]:
import tqdm

# Constants
PAD_LENGTH = 128
BATCH_SIZE = 256
EPOCHS = 5
bert_model = 'distilbert-base-uncased'

# Tokenizer
tokenizer = DistilBertTokenizer.from_pretrained(bert_model, do_lower_case=True, max_length=PAD_LENGTH, padding='max_length', truncation=True)

def tokenize(sentences, tokenizer, pad_length=PAD_LENGTH):

    if isinstance(sentences, str):
        inputs = tokenizer(
            sentences,
            padding='max_length',
            truncation=True,
            max_length=pad_length,
            return_attention_mask=True,
            return_tensors=None
        )

        return inputs['input_ids'], inputs['attention_mask'], None

    inputs = tokenizer(
        sentences,
        padding='max_length',
        truncation=True,
        max_length=pad_length,
        return_attention_mask=True
    )

    return inputs['input_ids'], inputs['attention_mask'], None



Let's read the data using Pandas.

In [ ]:
import pandas as pd

# Load data with only the necessary columns
train_df = pd.read_csv('hahackathon data/train.csv', usecols = ['text','humor_rating','offense_rating'])
dev_df = pd.read_csv('hahackathon data/dev.csv', usecols = ['text','humor_rating','offense_rating'])
test_df = pd.read_csv('hahackathon data/test.csv', usecols = ['text','humor_rating','offense_rating'])

# Drop the Nans
train_df = train_df.dropna()
dev_df = dev_df.dropna()
test_df = test_df.dropna()

Let's check a couple of examples.

In [ ]:
train_df

In [ ]:
# Get the post text
train_examples_list = train_df['text'].tolist()
dev_examples_list = dev_df['text'].tolist()
test_examples_list = test_df['text'].tolist()

# Get the humour rating for the regression task (we normalise, the values are between 0 and 5)
train_targets_list = (train_df['humor_rating']/5).tolist()
dev_targets_list = (dev_df['humor_rating']/5).tolist()
test_targets_list = (test_df['humor_rating']/5).tolist()

In [ ]:
def get_bert_inputs(examples_list, targets):
  input_ids=list()
  attention_masks=list()

  bert_inp=tokenize(examples_list, tokenizer)
  input_ids = bert_inp[0]
  attention_masks = bert_inp[1]

  return input_ids, attention_masks, targets

train_input_ids, train_attention_masks, train_targets = get_bert_inputs(train_examples_list, train_targets_list)
dev_input_ids, dev_attention_masks, dev_targets = get_bert_inputs(dev_examples_list, dev_targets_list)
test_input_ids, test_attention_masks, test_targets = get_bert_inputs(test_examples_list, test_targets_list)


**Task 1: Build a neural bag of words model using DistilBERT embeddings and the sigmoid activation on the output layer for the regression task.**

Investigate its performance using the Mean Squared Error (MSE) metric. We will use this metric as the loss function as well. We will also use the Adam optimiser with `learning_rate=2e-5`. This code is already provided.

*Hint*: You can reuse the code from Lab on Aspect-Based Sentiment Analysis (Model 2).

In [ ]:
import torch
import torch.nn as nn
from transformers import DistilBertModel

# Check if CUDA is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class GlobalAveragePooling1DMasked(nn.Module):

    def forward(self, x, mask):


        # Expand mask to match embedding dimension
        mask = mask.unsqueeze(-1).float()

        # Zero-out embeddings corresponding to padding tokens
        x = x * mask

        # Sum embeddings across the token dimension
        sum_x = torch.sum(x, dim=1)

        sum_mask = torch.sum(mask, dim=1)

        # Compute the average embedding (ignoring padding)
        return sum_x / sum_mask

# BERT Regression Model.

class BERTRegressionModel(nn.Module):

    def __init__(self, pad_length=128):
        super(BERTRegressionModel, self).__init__()

        # Load pretrained DistilBERT model
        self.bert = DistilBertModel.from_pretrained(bert_model)

        # Pooling layer to convert token embeddings into a single sentence vector
        self.global_avg_pool = GlobalAveragePooling1DMasked()

        # Hidden fully connected layer
        self.hidden_layer = nn.Linear(self.bert.config.hidden_size, 16)

        # Sigmoid activation function
        self.activation = nn.Sigmoid()

        # Outputs a single humour score
        self.output_reg = nn.Linear(16, 1)


    def forward(self, input_ids, attention_mask):

        bert_output = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )[0]

        pooled_output = self.global_avg_pool(bert_output, attention_mask)

        hidden_output = self.activation(self.hidden_layer(pooled_output))

        out_reg = self.activation(self.output_reg(hidden_output))

        return out_reg

model = BERTRegressionModel().to(device)

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)

print(model)

In [ ]:
print(model)

In [ ]:
# Dataset Class
class HumorDataset(Dataset):
    def __init__(self, input_ids, attention_masks, targets):
        self.input_ids = torch.tensor(input_ids)
        self.attention_masks = torch.tensor(attention_masks)
        self.targets = torch.tensor(targets)

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.attention_masks[idx],
            'target': self.targets[idx].float()
        }

In [ ]:
# Training setup
def train_model(model, train_loader, val_loader, device, epochs=EPOCHS, lr=2e-5):
    model.train()
    model.to(device)
    optimizer = AdamW(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for batch in tqdm.tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}'):
            optimizer.zero_grad()
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            targets = batch['target'].to(device)
            outputs = model(input_ids, attention_mask)

            loss = loss_fn(outputs.squeeze(), targets)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                targets = batch['target'].to(device)
                outputs = model(input_ids, attention_mask)
                loss = loss_fn(outputs.squeeze(), targets)
                val_loss += loss.item()

        print(f"Train Loss: {train_loss / len(train_loader):.4f}, Val Loss: {val_loss / len(val_loader):.4f}")


We evaluate our model on the test set.

In [ ]:
# Check if CUDA is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# Prepare DataLoader
train_dataset = HumorDataset(train_input_ids, train_attention_masks, train_targets)
dev_dataset = HumorDataset(dev_input_ids, dev_attention_masks, dev_targets)
test_dataset = HumorDataset(test_input_ids, test_attention_masks, test_targets)


train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
dev_loader = DataLoader(dev_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

# Instantiate model and train
model = BERTRegressionModel()
train_model(model, train_loader, dev_loader, device)

In [ ]:
from sklearn.metrics import mean_squared_error


# Evaluate the model
def evaluate_model(model, test_loader, device, multi_task=False):
    model.eval()
    model.to(device)
    loss_fn = nn.MSELoss()

    preds = []
    gt = []
    loss_list = []
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            targets = batch['target'].to(device)
            outputs = model(input_ids, attention_mask)
            #for multi_task setting we only evaluate the humor rating to make it inline with the other models.
            if multi_task:
              outputs = outputs[:,0]
              targets = targets[:,0].squeeze()
            preds.append(outputs.squeeze().cpu().numpy())
            gt.append(targets.cpu().numpy())
            loss_list.append(loss_fn(outputs.squeeze(), targets).cpu().numpy())

    preds = np.concatenate(preds)
    gt = np.concatenate(gt)

    mse = mean_squared_error(gt, preds)
    avg_loss = np.mean(loss_list)
    print(f'Test loss: {avg_loss:.4f}')
    print(f'Test Mean Squared Error: {mse:.4f}')
    return preds, gt

preds, test_targets = evaluate_model(model, test_loader, device)

Get the array of predictions here so that you can plot the outputs later.

## Predictive Distribution

We compute min, max and mean for the golden and predicted humour ratings.

In [ ]:
min(preds), max(preds), preds.mean()

In [ ]:
min(test_targets), max(test_targets), test_targets.mean()

In [ ]:
pd.Series(preds).hist()

In [ ]:
pd.Series(test_targets).hist()

Next, we plot the true vs predicted humour grade for our model.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
def get_pred_true_plot(preds, labels, title):
    limits = [labels.min(), labels.max()]
    fig, ax = plt.subplots()
    fig.set_dpi(150)
    ax.set_title(title)
    ax.scatter(labels, preds, marker='.')
    ax.plot(limits, limits, color="gray", linestyle=":")
    ax.set_xlabel('True Humour Grade')
    ax.set_ylabel('Predicted Humour Grade')
    sns.regplot(x=labels, y=preds, ax=ax, scatter_kws={"s": 5})
    plt.show()

get_pred_true_plot(preds, test_targets, 'True vs Predicted Humour Grade for DistilBERT Model')

Our regressor tends to smooth down the extreme rating values to make them closer to the mean.

# Feature Engineering & Data Augmentation

**Task 2: Preprocess the textual data with the Ekphrasis library following the standard pipeline https://github.com/cbaziotis/ekphrasis#text-pre-processing-pipeline. How does this affect the performance?**

*Hint*: You might not want to annotate terms in order to keep the same length of the input sentences (for this, do not use the parameter `annotate={"hashtag", "allcaps", "elongated", "repeated", 'emphasis', 'censored'}`).

In [ ]:
!pip install -q ekphrasis
!pip3 install -q emoji==0.6.0

In [ ]:
from ekphrasis.classes.preprocessor import TextPreProcessor
from ekphrasis.classes.tokenizer import SocialTokenizer

"""
Task 2

Begin
"""

# Ekphrasis preprocessor for social media text
text_processor = TextPreProcessor(
    # Normalize common noisy social media entities
    normalize=['url', 'email', 'user', 'percent', 'money', 'phone', 'time', 'date', 'number'],

    # Fix HTML tokens
    fix_html=True,

    # Use Twitter-based word segmentation and spelling correction
    segmenter="twitter",
    corrector="twitter",

    # Split hashtags into words
    unpack_hashtags=True,

    # Expand contractions
    unpack_contractions=True,

    # Keep elongated words instead of aggressively correcting them
    spell_correct_elong=False,

    tokenizer=SocialTokenizer(lowercase=True).tokenize,

    dicts=[]
)

"""
End Task 2
"""
new_train_examples_list = [" ".join(text_processor.pre_process_doc(example)) for example in train_examples_list]
new_dev_examples_list = [" ".join(text_processor.pre_process_doc(example)) for example in dev_examples_list]
new_test_examples_list = [" ".join(text_processor.pre_process_doc(example)) for example in test_examples_list]


In [ ]:
print("Original Text:", train_examples_list[0])
print("Preprocessed Text:", new_train_examples_list[0])

In [ ]:
train_input_ids, train_attention_masks, train_targets = get_bert_inputs(new_train_examples_list, train_targets_list)
dev_input_ids, dev_attention_masks, dev_targets = get_bert_inputs(new_dev_examples_list, dev_targets_list)
test_input_ids, test_attention_masks, test_targets = get_bert_inputs(new_test_examples_list, test_targets_list)


# Prepare DataLoader
train_dataset = HumorDataset(train_input_ids, train_attention_masks, train_targets)
dev_dataset = HumorDataset(dev_input_ids, dev_attention_masks, dev_targets)
test_dataset = HumorDataset(test_input_ids, test_attention_masks, test_targets)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
dev_loader = DataLoader(dev_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

model = BERTRegressionModel()
train_model(model, train_loader, dev_loader, device)

In [ ]:
preds, test_targets = evaluate_model(model, test_loader, device)

Your results may be different depending on the implementation but typically special preprocessing does not drastically change the performance for this task.

**Task 3: Augment the training data twice by changing the original data via two methods from the Nlpaug (https://github.com/makcedward/nlpaug) library: (a) synonym replacement from WordNet; (b) deletion of random words. Comment on which method gives the best performance.**

*Hint*: Use the Synonym Augmenter and Random Word Augmenter (Delete word randomly) classes as follows:
```
aug = naw.SynonymAug(aug_src='wordnet')
augmented_text = aug.augment(text)

aug = naw.RandomWordAug()
augmented_text = aug.augment(text)
```





For more examples check https://github.com/makcedward/nlpaug/blob/master/example/textual_augmenter.ipynb

In [ ]:
!pip install nlpaug

In [ ]:
# Get the data again to apply augmentation
train_examples_list = train_df['text'].tolist()
train_targets_list = (train_df['humor_rating']/5).tolist()

In [ ]:
import nlpaug.augmenter.word as naw
import nltk
nltk.download('averaged_perceptron_tagger_eng')

# Get the original data to use for augmentation
orig_train_examples = train_examples_list
orig_train_targets = train_targets_list

# Synonym replacement augmenter
aug = naw.SynonymAug(aug_src='wordnet')

# Create augmented texts
augmented_examples = []
for text in orig_train_examples:
    aug_text = aug.augment(text)
    if isinstance(aug_text, list):
        aug_text = aug_text[0]
    augmented_examples.append(aug_text)

# Combine original + augmented data
train_examples_list = orig_train_examples + augmented_examples
train_targets_list = orig_train_targets + orig_train_targets


In [ ]:
import nlpaug.augmenter.word as naw
'''
#if you want to run this model comment out synonym replacement block just above it and then run Random deletion model.
# Reset to the original training data
orig_train_examples = train_df['text'].tolist()
orig_train_targets = (train_df['humor_rating'] / 5).tolist()

# Random word deletion augmenter
aug = naw.RandomWordAug(action='delete')

# Create augmented texts
augmented_examples = []
for text in orig_train_examples:
    aug_text = aug.augment(text)
    if isinstance(aug_text, list):
        aug_text = aug_text[0]
    augmented_examples.append(aug_text)

# Combine original + augmented data
train_examples_list = orig_train_examples + augmented_examples
train_targets_list = orig_train_targets + orig_train_targets
'''


In [ ]:
print(train_examples_list[-2])
print(train_examples_list[4930])

In [ ]:
train_input_ids, train_attention_masks, train_targets = get_bert_inputs(train_examples_list, train_targets_list)
dev_input_ids, dev_attention_masks, dev_targets = get_bert_inputs(dev_examples_list, dev_targets_list)
test_input_ids, test_attention_masks, test_targets = get_bert_inputs(test_examples_list, test_targets_list)

# Prepare DataLoader
train_dataset = HumorDataset(train_input_ids, train_attention_masks, train_targets)
dev_dataset = HumorDataset(dev_input_ids, dev_attention_masks, dev_targets)
test_dataset = HumorDataset(test_input_ids, test_attention_masks, test_targets)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
dev_loader = DataLoader(dev_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

We have now augmented the original data twice:

In [ ]:
print("Training examples before augmentation:")
print(len(train_df['text'].tolist()))
print("Training examples after augmentation:")
print(len(train_examples_list))

In [ ]:
model = BERTRegressionModel()
train_model(model, train_loader, dev_loader, device)

In [ ]:
preds, test_targets = evaluate_model(model, test_loader, device)

Your results may be different depending on the implementation but typically there are no drastic differences between the augmentation setups.

# Ensembled BERT Model

In this section you will train and evaluate an **ensemble** of BERT models.

We define the hyperparameters, including the number of models we want to ensemble (RERUNS=3, i.e., 3 models).

**Task 4: Train three DistilBERT models, get their predictions on the test set, take the mean of those predictions and evaluate this ensembled prediction. Comment on the resulting performance.**

We create three models in a loop, set a new random seed before creating each of them (`set_random_seed(seed=random.randint(0, 500))`) and accumulate predictions per model in a list.

In [ ]:
# Get the train data again to avoid any confusion
train_examples_list = train_df['text'].tolist()
train_targets_list = (train_df['humor_rating']/5).tolist()

train_input_ids, train_attention_masks, train_targets = get_bert_inputs(train_examples_list, train_targets_list)

# Prepare DataLoader
train_dataset = HumorDataset(train_input_ids, train_attention_masks, train_targets)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)


In [ ]:
RERUNS = 3
all_model_preds = list()

for i in range(RERUNS):

  set_random_seed(seed=random.randint(0, 500))

  """
  Task 4

  Begin
  """

  # Creating  model
  model = BERTRegressionModel()

  # Training the model on the original data
  train_model(model, train_loader, dev_loader, device)

  # Getting predictions on the test set
  preds, gt = evaluate_model(model, test_loader, device)

  # Store the model's predictions for ensembling
  all_model_preds.append(preds)

  """
  End Task 4
  """

In [ ]:
from sklearn.metrics import classification_report, mean_squared_error

mean_preds = np.mean(np.array(all_model_preds), axis=0)
ensemble_mse = mean_squared_error(test_targets, mean_preds)

print('Ensemble Test MSE : {:.4f}'.format(ensemble_mse))

Your results may be different depending on the implementation but typically ensembling slightly improves the performance for this task.

# Multi-task Learning with BERT

**Task 5: Train a multi-task (MTL) model with the additional regression task of predicting the offense rating. The code to fetch these data is provided below. Comment on the resulting performance for the two models (single-task from task 1 and MTL).**

*Hint*: The MTL model will have two identical output layers (one for predicting humour rating, the other to predict offense rating). The layers need to be concatenated before applying the loss fuctions.

In [ ]:
import gc
if 'model' in locals():
  del model
gc.collect()
with torch.no_grad():
    torch.cuda.empty_cache()

In [ ]:
class BertMultitaskModel(nn.Module):
    """
    Task 5

    Begin
    """
    def __init__(self, pad_length=128):
        super(BertMultitaskModel, self).__init__()

        # Shared DistilBERT encoder
        self.bert = DistilBertModel.from_pretrained(bert_model)

        # Pooling layer
        self.global_avg_pool = GlobalAveragePooling1DMasked()

        # Shared hidden layer
        self.hidden_layer = nn.Linear(self.bert.config.hidden_size, 16)
        self.activation = nn.Sigmoid()

        # Two regression heads
        # Head 1 → humour rating
        self.output_reg1 = nn.Linear(16, 1)

        # Head 2 → offense rating
        self.output_reg2 = nn.Linear(16, 1)


    def forward(self, input_ids, attention_mask):

        # Getting token embeddings from DistilBERT
        bert_output = self.bert(input_ids=input_ids, attention_mask=attention_mask)[0]

        # Pool embeddings across tokens
        pooled_output = self.global_avg_pool(bert_output, attention_mask)

        # Shared hidden representation
        hidden_output = self.activation(self.hidden_layer(pooled_output))

        # Predict humour
        out_reg1 = self.activation(self.output_reg1(hidden_output))

        # Predict offense
        out_reg2 = self.activation(self.output_reg2(hidden_output))

        # Concatenate predictions
        return torch.cat([out_reg1, out_reg2], dim=1)
    """
    End Task 5
    """

In [ ]:
model = BertMultitaskModel(pad_length=PAD_LENGTH).to(device)


In [ ]:
train_examples_list = train_df['text'].tolist()
train_targets_list = (train_df['humor_rating']/5).tolist()

train_input_ids, train_attention_masks, train_targets = get_bert_inputs(train_examples_list, train_targets_list)

# Get the offense ratings for the second regression task (we normalise them as well)
train_targets2_list = (train_df['offense_rating']/5).tolist()
dev_targets2_list = (dev_df['offense_rating']/5).tolist()
test_targets2_list = (test_df['offense_rating']/5).tolist()

train_targets2 = np.array(train_targets2_list)
dev_targets2 = np.array(dev_targets2_list)
test_targets2 = np.array(test_targets2_list)

# Prepare DataLoader
train_targets_mix = [[i[0],i[1]] for i in zip(train_targets,train_targets2)]
train_dataset = HumorDataset(train_input_ids, train_attention_masks, train_targets_mix)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
dev_targets_mix = [[i[0],i[1]] for i in zip(dev_targets,dev_targets2)]
dev_dataset = HumorDataset(dev_input_ids, dev_attention_masks, dev_targets_mix)
dev_loader = DataLoader(dev_dataset, batch_size=BATCH_SIZE)
test_targets_mix = [[i[0],i[1]] for i in zip(test_targets,test_targets2)]
test_dataset = HumorDataset(test_input_ids, test_attention_masks, test_targets_mix)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

In [ ]:
# Instantiate model and train
train_model(model, train_loader, dev_loader, device)

In [ ]:
preds, test_targets = evaluate_model(model, test_loader, device, multi_task=True)

Your results may be different depending on the implementation but typically MTL slightly improves the performance over the single-task model trained in similar conditions.